---
title: "13. Operations on Azure"
description: "Operate the deployed platform with results state, ACA execution status, readiness probes, alerts, dashboard controls, and promotion rollback."
categories: []
---

Operations is where the shared contract becomes evidence. Azure supplies the control plane, identity, logging, and alerting adapters; the workload still reports the same results statuses, model version identity, readiness state, and prediction response that the local Compose golden path checks.


## Four signals, one runbook

Use different signals for different questions:

| Question | Evidence |
|---|---|
| Did ACA start and finish the process? | `az containerapp job execution show`, polled until `Succeeded`, `Failed`, `Stopped`, or timeout |
| Did the application complete its work? | Parent/child rows in the `results` database and dashboard API |
| Which model is serving? | `/readyz` returns `status=ready`, `model_name`, and exact `model_version` |
| Did inference work? | `/v1/predictions` returns a prediction with the same model version |

A green Job execution is necessary but not sufficient: a process can exit zero after recording an incorrect outcome. That is why the smoke adapter waits for the cloud execution and then checks the behavioral result row. The local golden path applies the same assertions through its runner.


## Alerts and dashboards

The `observability` module creates four Log Analytics scheduled-query rules:

- a failed ACA Job execution;
- a missed scheduled-run window;
- permanent child failures above the configured threshold;
- a batch circuit breaker event.

Action groups are optional, so a development deployment can create rules without paging anyone. The queries for application traces and results-derived signals are intentionally visible in `infra/modules/observability/alerts.tf`; tune them when diagnostic routing and retention are finalized.

The dashboard App is a catalog and launcher, not an executor. In Azure it uses Easy Auth for the human identity, `id-dashboard` for read-only results access, and a scoped custom role to start ACA Jobs. The local dashboard uses the runner adapter instead. Both show the same results rows and expose the same health/readiness surfaces.


## Promote, verify, rollback

Promotion is a registry operation followed by a consumer update:

~~~bash
python demo/promote.py --tracking-uri https://<mlflow-app> \\
  --backend aca --version 3 \\
  --resource-group <resource-group> --app-name <serving-app> --execute
~~~

The registry alias is standardized as `production`. Serving is never left on that floating alias: the ACA App receives the exact `MODEL_VERSION=3` value, starts a new revision, and must pass `/readyz` before it is considered ready. A rollback flips `production` to the known-good version and repins the App to that version.

Keep the tracking URI explicit in automation. `--tracking-uri` and `MLFLOW_TRACKING_URI` select the local Compose MLflow or cloud MLflow without changing promotion logic.


## Cloud smoke tests and operating limits

Run the Azure adapter after the deployment:

~~~bash
./deploy/smoke-tests.sh --tf-vars infra/environments/dev.tfvars
~~~

PowerShell users can run the matching `.ps1` script. The local-only `demo/golden_path.py` is not passed an ACA backend; this separation keeps local runner behavior and Azure control-plane behavior honest while reusing the same assertions.

The remaining operational boundaries are explicit: public endpoints are an MVP networking choice, alert queries need production diagnostic tuning, and the cloud evaluator needs an accessible JSONL dataset plus a Key Vault model credential. Those are deployment and operations concerns, not reasons to fork the shared model or job code.

Next: [14 — Multi-GPU training](14-multi-gpu-training.ipynb) covers the admission-gated exception path.
